# Preliminary CoT-driven RAG test


In [1]:
import os

# from dotenv import load_dotenv
import sys

import pandas as pd

project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)
# # Load secrets from .env if present (expects ANTHROPIC_API_KEY when using Claude)
load_dotenv()
print("ANTHROPIC set?", "ANTHROPIC_API_KEY" in os.environ)

from mozzarellm.configs import DEFAULT_ANTHROPIC_CONFIG  #

# os.environ["ANTHROPIC_API_KEY"] = "placeholder"

NameError: name 'load_dotenv' is not defined

In [ ]:
# Imports from the package
from mozzarellm import analyze_gene_clusters, reshape_to_clusters
from mozzarellm.prompts import ROBUST_CLUSTER_PROMPT, ROBUST_SCREEN_CONTEXT

# from mozzarellm.configs import DEFAULT_ANTHROPIC_CONFIG
# Load sample data used in the example notebook
notebook_dir = os.path.abspath(os.path.dirname("__file__"))
project_root = os.path.abspath(os.path.join(notebook_dir, ".."))
sample_csv = os.path.join(project_root, "examples", "sample_data.csv")
sample_data = pd.read_csv(sample_csv)

cluster_df, gene_features = reshape_to_clusters(
    input_df=sample_data, uniprot_col="uniprot_function", verbose=True
)
display(cluster_df.head())
display(gene_features.head())

Using provided DataFrame with 140 rows
Found 140 genes across 6 clusters
Extracting gene features from uniprot_function column


,cluster_id,genes
0,21,AATF;ABT1;BYSL;BMS1;C1orf131;EIF3M;EIF4A1;ESF1...
1,37,SRSF3;PDPK1;RICTOR;RPTOR;SEH1L;SGF29;PRKAR1A;P...
2,121,CCDC174;FAM32A;GABPA;SP2;N6AMT1;SETD2;SON;POU5...
3,149,KRAS;BRAF;NDUFV2;NDUFA6;NDUFC1;RAD23B;SNAPC1;N...
4,167,POMP;PSMA2;PSMB7;PSMB3;PSMA7;PSMB2;PSMA1;PSMA4...


,gene_symbol,uniprot_function
0,AATF,"Part of the small subunit (SSU) processome, fi..."
1,ABT1,Could be a novel TATA-binding protein (TBP) wh...
2,BYSL,Required for processing of 20S pre-rRNA precur...
3,BMS1,GTPase required for the synthesis of 40S ribos...
4,C1orf131,"Part of the small subunit (SSU) processome, fi..."


In [ ]:
%load_ext autoreload
%autoreload 2

import importlib
import os
import sys

# Ensure project root is on path (adjust if needed)
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
if project_root not in sys.path:
    sys.path.insert(0, project_root)

import mozzarellm.utils.cluster_analyzer as ca

import mozzarellm
import mozzarellm.utils.prompt_factory as pf
import mozzarellm.utils.retrieval as rt

importlib.reload(rt)
importlib.reload(pf)
importlib.reload(ca)
importlib.reload(mozzarellm)

import inspect

print(inspect.signature(analyze_gene_clusters))

(input_file=None, input_df=None, input_sep=',', gene_column='genes', gene_sep=';', cluster_id_column='cluster_id', model_name=None, config_path=None, config_dict=None, screen_context_path=None, screen_context=None, cluster_analysis_prompt_path=None, cluster_analysis_prompt=None, gene_annotations_path=None, gene_annotations_df=None, batch_size=1, start_idx=0, end_idx=None, output_file=None, save_outputs=True, outputs_to_generate=['json', 'clusters', 'flagged_genes'], use_retrieval=False, knowledge_dir=None, retriever_k=10, cot_instructions=None)


In [ ]:
# Minimal structured CoT instructions (compact, auditable)
cot = """
1) List 2-3 candidate pathways and key mechanisms.
2) Classify genes as ESTABLISHED / UNCHARACTERIZED / NOVEL_ROLE with 1-line rationale.
3) Cite retrieved snippet indices [1..] that support each decision where possible.
4) Note contradictions/gaps; adjust confidence before final JSON.
"""
print(cot)


1) List 2-3 candidate pathways and key mechanisms.
2) Classify genes as ESTABLISHED / UNCHARACTERIZED / NOVEL_ROLE with 1-line rationale.
3) Cite retrieved snippet indices [1..] that support each decision where possible.
4) Note contradictions/gaps; adjust confidence before final JSON.



In [ ]:
# Enable prelim retrieval using local knowledge folder
knowledge_dir = os.path.join(project_root, "data", "knowledge")
print("Knowledge dir:", knowledge_dir, "exists?", os.path.isdir(knowledge_dir))

Knowledge dir: c:\Users\alexa\WORKSPACE\UROP\mozzarellm\data\knowledge exists? True


In [ ]:
# Run with Claude + preliminary RAG
results = analyze_gene_clusters(
    input_df=cluster_df,
    model_name="claude-3-7-sonnet-20250219",
    config_dict=DEFAULT_ANTHROPIC_CONFIG,
    screen_context=ROBUST_SCREEN_CONTEXT,
    cluster_analysis_prompt=ROBUST_CLUSTER_PROMPT,
    gene_annotations_df=gene_features,
    batch_size=1,
    save_outputs=False,
    outputs_to_generate=["json", "clusters", "flagged_genes"],
    # New flags
    use_retrieval=True,
    knowledge_dir=knowledge_dir,
    retriever_k=10,
    cot_instructions=cot,
)
list(results.keys())

Loaded data with 6 rows and columns: ['cluster_id', 'genes']
Created annotations dictionary with 140 entries from DataFrame


Processing clusters:   0%|          | 0/6 [00:00<?, ?it/s]

Using provided template string
Appending output format instructions to template
Added 39 gene feature descriptions to prompt


Processing clusters:  17%|█▋        | 1/6 [00:19<01:35, 19.02s/it]

Using provided template string
Appending output format instructions to template
Added 33 gene feature descriptions to prompt


...

Using provided template string
Appending output format instructions to template
Added 16 gene feature descriptions to prompt


Processing clusters:  83%|████████▎ | 5/6 [01:29<00:16, 16.78s/it]

Using provided template string
Appending output format instructions to template
Added 11 gene feature descriptions to prompt


Processing clusters: 100%|██████████| 6/6 [01:43<00:00, 17.20s/it]
INFO:cluster_analysis_20251019_191323.log:Completed analysis for 6 clusters without saving to disk


['clusters_dict', 'json_data', 'cluster_df', 'gene_df']

In [ ]:
# Inspect outputs
results.get("cluster_df")

,cluster_id,cluster_biological_process,pathway_confidence_level,cluster_importance_score,follow_up_suggestion,established_genes,established_gene_count,uncharacterized_genes,uncharacterized_gene_count,novel_role_genes,...,total_gene_count,highest_unchar_importance,average_unchar_importance,high_unchar_genes,high_unchar_gene_count,highest_novel_role_importance,average_novel_role_importance,high_novel_role_genes,high_novel_role_gene_count,all_cluster_genes
0,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,17,NOC4L;NOL10;RRP12,3,INO80;DYRK1A;HK2;NEPRO;ZCCHC9,...,25,7,6.67,,0,8,6.60,INO80:8,1,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...
1,37,mTOR signaling pathway,High,2.10,This cluster shows a strong enrichment for com...,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...,9,C7orf26,1,SLC4A7;TRAPPC8;TRAPPC3;TRAPPC11,...,14,5,5.00,,0,7,5.75,,0,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...
2,121,Transcriptional regulation via chromatin modif...,High,2.64,Cluster 121 reveals a coherent transcriptional...,MYC;MAX;GABPA;GABPB1;SP2;SETD2;ZMYND8;KEAP1,8,CCDC174;POU5F1B;ZBTB11,3,N6AMT1;SON;ILF3;HNRNPM;FAM32A;DDA1;RAB11FIP4;M...,...,22,7,6.67,,0,8,5.91,N6AMT1:8,1,MYC;MAX;GABPA;GABPB1;SP2;SETD2;ZMYND8;KEAP1;CC...
3,149,Mitochondrial oxidative phosphorylation and el...,High,2.10,This cluster shows a strong enrichment for mit...,NDUFV2;NDUFA6;NDUFC1;DMAC1;CYC1;UQCRFS1;ATP5ME...,12,,0,KRAS;BRAF;RAD23B;UBAC2;NINL;SNAPC1;LSM11,...,19,0,0.00,,0,7,4.86,,0,NDUFV2;NDUFA6;NDUFC1;DMAC1;CYC1;UQCRFS1;ATP5ME...
4,167,Proteasome-mediated protein degradation pathway,High,2.10,Cluster 167 shows a highly coherent biological...,POMP;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7...,14,,0,AKIRIN2;UBE2R2,...,16,0,0.00,,0,7,6.00,,0,POMP;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7...
5,197,N6-methyladenosine (m6A) RNA modification pathway,Medium,1.40,Cluster 197 shows a medium-confidence signatur...,METTL3;METTL14,2,PTCHD4,1,HNRNPD;VRK1;MCM3,...,6,6,6.00,,0,7,6.00,,0,METTL3;METTL14;PTCHD4;HNRNPD;VRK1;MCM3


In [ ]:
results.get("gene_df")

,gene_name,gene_description,gene_importance_score,cluster_id,cluster_biological_process,pathway_confidence_level,cluster_importance_score,follow_up_suggestion,established_genes,established_gene_count,uncharacterized_genes,uncharacterized_gene_count,novel_role_genes,novel_role_gene_count,gene_category
8,INO80,Well-characterized as an ATP-dependent chromat...,8,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,17,NOC4L;NOL10;RRP12,3,INO80;DYRK1A;HK2;NEPRO;ZCCHC9,5,novel_role
17,N6AMT1,Known as a methyltransferase that modifies his...,8,121,Transcriptional regulation via chromatin modif...,High,2.64,Cluster 121 reveals a coherent transcriptional...,MYC;MAX;GABPA;GABPB1;SP2;SETD2;ZMYND8;KEAP1,8,CCDC174;POU5F1B;ZBTB11,3,N6AMT1;SON;ILF3;HNRNPM;FAM32A;DDA1;RAB11FIP4;M...,11,novel_role
9,DYRK1A,Established as a dual-specificity kinase invol...,7,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,17,NOC4L;NOL10;RRP12,3,INO80;DYRK1A;HK2;NEPRO;ZCCHC9,5,novel_role
10,HK2,Well-characterized as a glycolytic enzyme that...,7,21,Ribosome biogenesis and pre-rRNA processing,High,2.64,Cluster 21 shows a strong signature of ribosom...,AATF;BYSL;BMS1;C1orf131;DIMT1;DDX52;DDX47;NOP1...,17,NOC4L;NOL10;RRP12,3,INO80;DYRK1A;HK2;NEPRO;ZCCHC9,5,novel_role
18,SON,Well-characterized in RNA splicing but its DNA...,7,121,Transcriptional regulation via chromatin modif...,High,2.64,Cluster 121 reveals a coherent transcriptional...,MYC;MAX;GABPA;GABPB1;SP2;SETD2;ZMYND8;KEAP1,8,CCDC174;POU5F1B;ZBTB11,3,N6AMT1;SON;ILF3;HNRNPM;FAM32A;DDA1;RAB11FIP4;M...,11,novel_role
22,DDA1,Functions in E3 ubiquitin ligase complexes but...,7,121,Transcriptional regulation via chromatin modif...,High,2.64,Cluster 121 reveals a coherent transcriptional...,MYC;MAX;GABPA;GABPB1;SP2;SETD2;ZMYND8;KEAP1,8,CCDC174;POU5F1B;ZBTB11,3,N6AMT1;SON;ILF3;HNRNPM;FAM32A;DDA1;RAB11FIP4;M...,11,novel_role
13,SLC4A7,SLC4A7 is a sodium-bicarbonate cotransporter p...,7,37,mTOR signaling pathway,High,2.10,This cluster shows a strong enrichment for com...,MTOR;RPTOR;RICTOR;MLST8;RHEB;PDPK1;SEH1L;WDR24...,9,C7orf26,1,SLC4A7;TRAPPC8;TRAPPC3;TRAPPC11,4,novel_role
28,KRAS,KRAS is well-characterized in MAPK signaling b...,7,149,Mitochondrial oxidative phosphorylation and el...,High,2.10,This cluster shows a strong enrichment for mit...,NDUFV2;NDUFA6;NDUFC1;DMAC1;CYC1;UQCRFS1;ATP5ME...,12,,0,KRAS;BRAF;RAD23B;UBAC2;NINL;SNAPC1;LSM11,7,novel_role
35,AKIRIN2,AKIRIN2 has established roles in immunity and ...,7,167,Proteasome-mediated protein degradation pathway,High,2.10,Cluster 167 shows a highly coherent biological...,POMP;PSMA1;PSMA2;PSMA3;PSMA4;PSMA5;PSMA6;PSMA7...,14,,0,AKIRIN2;UBE2R2,2,novel_role
37,HNRNPD,Known RNA-binding protein that regulates mRNA ...,7,197,N6-methyladenosine (m6A) RNA modification pathway,Medium,1.40,Cluster 197 shows a medium-confidence signatur...,METTL3;METTL14,2,PTCHD4,1,HNRNPD;VRK1;MCM3,3,novel_role
